# 05. Segmentation & RFM

In [ ]:
import pandas as pd
import os
import kagglehub
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

dataset_path = kagglehub.dataset_download('radistaleks/synthetic-bank-transactions')
categories    = pd.read_csv(os.path.join(dataset_path, 'categories.csv'))
clients       = pd.read_csv(os.path.join(dataset_path, 'clients.csv'))
subscriptions = pd.read_csv(os.path.join(dataset_path, 'subscriptions.csv'))
transactions  = pd.read_csv(os.path.join(dataset_path, 'transactions.csv'))

In [ ]:
clients['registration_date'] = pd.to_datetime(clients['registration_date'])
clients['birthdate']         = pd.to_datetime(clients['birthdate'])
subscriptions['date_start']  = pd.to_datetime(subscriptions['date_start'])
subscriptions['date_end']    = pd.to_datetime(subscriptions['date_end'])
transactions['date']         = pd.to_datetime(transactions['date'], format='%Y-%m-%d %H:%M:%S')

clients = clients.fillna(0)
subscriptions['product_company'] = subscriptions['product_company'].fillna('Неизвестно')
transactions['product_company']  = transactions['product_company'].fillna('Неизвестно')

cat_map = dict(zip(categories['id'], categories['name']))
transactions['category_name'] = transactions['product_category'].map(cat_map)

N = len(clients)
SNAPSHOT = pd.Timestamp('2020-12-31')

## 1. RFM

In [ ]:
rfm = transactions.groupby('client_id').agg(
    last_date = ('date', 'max'),
    frequency = ('amount', 'count'),
    monetary  = ('amount', 'sum')
).reset_index()
rfm['recency'] = (SNAPSHOT - rfm['last_date']).dt.days
rfm = rfm.drop(columns='last_date')
rfm.describe()

In [ ]:
# скоринг: 5 = лучший, 1 = худший
rfm['r_score'] = pd.qcut(rfm['recency'],  q=5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary'].rank(method='first'),  q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['rfm_score'] = rfm['r_score'] + rfm['f_score'] + rfm['m_score']
rfm.head()

In [ ]:
def get_segment(row):
    r, f, m = row['r_score'], row['f_score'], row['m_score']
    if r >= 4 and f >= 4 and m >= 4:   return 'Champions'
    elif r >= 3 and f >= 3:             return 'Loyal'
    elif r >= 4 and f <= 2:             return 'New / Promising'
    elif r <= 2 and f >= 3:             return 'At Risk'
    elif r <= 2 and f <= 2 and m <= 2:  return 'Lost'
    else:                               return 'Average'

rfm['segment'] = rfm.apply(get_segment, axis=1)
rfm['segment'].value_counts()

In [ ]:
rfm['segment'].value_counts().plot(kind='bar', figsize=(10, 4), title='RFM-сегменты', rot=20)

In [ ]:
rfm.groupby('segment')[['recency', 'frequency', 'monetary', 'rfm_score']].mean().round(0).sort_values('rfm_score', ascending=False)

In [ ]:
# scatter: frequency vs monetary, цвет = recency
plt.figure(figsize=(12, 6))
sc = plt.scatter(rfm['frequency'], rfm['monetary'], c=rfm['recency'], cmap='RdYlGn_r', alpha=0.7, s=30)
plt.colorbar(sc, label='Recency (дней)')
plt.xlabel('Frequency')
plt.ylabel('Monetary (руб)')
plt.title('RFM: Frequency vs Monetary (цвет = Recency)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, ['recency', 'frequency', 'monetary']):
    rfm.boxplot(column=col, by='segment', ax=ax, rot=30)
    ax.set_title(col)
    ax.set_xlabel('')
plt.suptitle('RFM-метрики по сегментам')
plt.tight_layout()

## 2. Пол

In [ ]:
clients['gender'].value_counts()

In [ ]:
txn_g = transactions.merge(clients[['id', 'gender']], left_on='client_id', right_on='id', how='left')

txn_g.groupby('gender').agg(
    users=('client_id','nunique'), count=('amount','count'),
    total=('amount','sum'), avg=('amount','mean'), median=('amount','median')
)

In [ ]:
# средний чек по категории и полу
gender_cat = txn_g.groupby(['category_name', 'gender'])['amount'].mean().unstack()
gender_cat.sort_values('F', ascending=False).head(15)

In [ ]:
gender_cat.sort_values('F', ascending=False).head(15).plot(
    kind='barh', figsize=(12, 8), title='Средний чек по категориям: М vs Ж'
)

## 3. Возраст

In [ ]:
clients['age'] = (SNAPSHOT - clients['birthdate']).dt.days // 365
clients['age_group'] = pd.cut(
    clients['age'], bins=[17, 25, 35, 45, 55, 100],
    labels=['18-25', '26-35', '36-45', '46-55', '55+']
)
clients['age_group'].value_counts().sort_index()

In [ ]:
txn_a = transactions.merge(clients[['id', 'age_group']], left_on='client_id', right_on='id', how='left')

txn_a.groupby('age_group', observed=True).agg(
    users=('client_id','nunique'), count=('amount','count'),
    total=('amount','sum'), avg=('amount','mean')
).assign(txn_per_user=lambda d: d['count'] / d['users'])

In [ ]:
txn_a.groupby('age_group', observed=True)['amount'].mean().sort_index().plot(
    kind='bar', figsize=(10, 4), title='Средний чек по возрастной группе', rot=0
)

## 4. Портфель продуктов

In [ ]:
active_subs = subscriptions[subscriptions['date_end'].isna()]

portfolio = clients[['id', 'credit', 'deposit']].copy()
portfolio['has_sub']      = portfolio['id'].isin(active_subs['client_id']).astype(int)
portfolio['has_music']    = portfolio['id'].isin(active_subs[active_subs['product_category'] == 4]['client_id']).astype(int)
portfolio['n_products']   = (portfolio['credit'] == 1).astype(int) + (portfolio['deposit'] == 1).astype(int) + portfolio['has_sub']
portfolio['n_products'].value_counts().sort_index()

In [ ]:
txn_p = transactions.merge(portfolio[['id', 'n_products', 'has_music']], left_on='client_id', right_on='id', how='left')

txn_p.groupby('n_products').agg(
    users=('client_id','nunique'), avg=('amount','mean'), total=('amount','sum'), count=('amount','count')
).assign(txn_per_user=lambda d: d['count'] / d['users'], arpu=lambda d: d['total'] / d['users'])

In [ ]:
# музыкальная подписка — влияет ли на транзакционную активность?
txn_p.groupby('has_music').agg(
    users=('client_id','nunique'), avg=('amount','mean'), count=('amount','count'), total=('amount','sum')
).assign(txn_per_user=lambda d: d['count'] / d['users'])

## 5. K-Means по категориям трат

In [ ]:
# матрица клиент × категория (доля от суммарных трат)
cat_matrix = transactions.pivot_table(
    values='amount', index='client_id', columns='category_name', aggfunc='sum', fill_value=0
)
cat_norm = cat_matrix.div(cat_matrix.sum(axis=1), axis=0)
cat_norm.shape

In [ ]:
X = StandardScaler().fit_transform(cat_norm)

# elbow
inertias = [KMeans(n_clusters=k, random_state=42, n_init=10).fit(X).inertia_ for k in range(2, 10)]
pd.Series(inertias, index=range(2, 10)).plot(marker='o', figsize=(10, 4), title='Elbow: выбор числа кластеров')

In [ ]:
K = 4
cat_norm['cluster'] = KMeans(n_clusters=K, random_state=42, n_init=10).fit_predict(X)
cat_norm['cluster'].value_counts().sort_index()

In [ ]:
# профили кластеров — топ-категории
profiles = cat_norm.groupby('cluster').mean().drop(columns='cluster', errors='ignore')
profiles.T.plot(kind='bar', figsize=(18, 6), title='Доля трат по категориям в кластерах')
plt.xticks(rotation=45, ha='right')

In [ ]:
for cl in range(K):
    top = profiles.loc[cl].sort_values(ascending=False).head(5)
    print(f'\nКластер {cl}:')
    print(top.round(3).to_string())

In [ ]:
# RFM + кластер
cluster_map = cat_norm[['cluster']].reset_index()
rfm_cl = rfm.merge(cluster_map, on='client_id', how='left')
rfm_cl.groupby('cluster')[['recency', 'frequency', 'monetary', 'rfm_score']].mean().round(0)